In [11]:
!git clone https://github.com/joaovictorferian/chat-bot-goodwe-files.git

fatal: destination path 'chat-bot-goodwe-files' already exists and is not an empty directory.


In [13]:
!git -C /content/chat-bot-goodwe-files pull #Rodar apenas se novos arquivos foram adicionados no repositório

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 5.54 MiB | 9.20 MiB/s, done.
From https://github.com/joaovictorferian/chat-bot-goodwe-files
   15ebd0b..1af7167  main       -> origin/main
Updating 15ebd0b..1af7167
Fast-forward
 goodwe_docs/GW_ESS Troubleshooting Guide-EMEA_EN.pdf  | Bin 0 -> 653902 bytes
 goodwe_docs/GW_HCA Series_User Manual-EN.pdf          | Bin 0 -> 5752608 bytes
 .../Goodwe HCA Series Quick Installation Manual 2.pdf | Bin 0 -> 5823995 bytes
 3 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 goodwe_docs/GW_ESS Troubleshooting Guide-EMEA_EN.pdf
 create mode 100644 goodwe_docs/GW_HCA Series_User Manual-EN.pdf
 create mode 100644 goodwe_docs/Goodwe HCA Series Quick Installation Manual 2.pdf


In [12]:
!pip install -q \
    langchain==0.1.20 \
    langchain-openai==0.1.6 \
    langchain-community==0.0.38 \
    langchain-text-splitters \
    faiss-cpu \
    pypdf \
    tiktoken \
    ipywidgets

In [14]:
import os
from google.colab import userdata

In [15]:
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

DOCS_DIR   = '/content/chat-bot-goodwe-files/goodwe_docs'

os.makedirs(DOCS_DIR, exist_ok=True)

In [16]:
import glob
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
pdf_paths = glob.glob(os.path.join(DOCS_DIR, '**/*.pdf'), recursive=True)

if not pdf_paths:
    print('Nenhum PDF encontrado em', DOCS_DIR)
    print('   Adicione os manuais GoodWe na pasta e rode novamente.')
else:
    print(f'{len(pdf_paths)} PDF(s) encontrado(s):')
    for p in pdf_paths:
        print(f'   • {os.path.basename(p)}')

    all_docs = []
    for path in pdf_paths:
        loader = PyPDFLoader(path)
        docs = loader.load()
        for doc in docs:
            doc.metadata['source_file'] = os.path.basename(path)
        all_docs.extend(docs)
        print(f'{os.path.basename(path)} — {len(docs)} páginas')

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    chunks = splitter.split_documents(all_docs)
    print(f'\n{len(chunks)} chunks gerados a partir de {len(all_docs)} páginas')

10 PDF(s) encontrado(s):
   • 1CC - EV CHALLENGE GOODWE+FIAP 2026.pdf
   • Goodwe HCA Series Quick Installation Manual.pdf
   • Goodwe HCA Series Quick Installation Manual 2.pdf
   • GW_SEMS Portal APP_User Manual-EN.pdf
   • GW_HCA-G2_Datasheet-EN.pdf
   • GW_ESS Troubleshooting Guide-EMEA_EN.pdf
   • Apresentação Challenge FIAP4+GoodWe 2026.pdf
   • FIAP_EV Challenge_2026_Mentoria1.pdf
   • Goodwe Hca Series User Manual.pdf
   • GW_HCA Series_User Manual-EN.pdf
1CC - EV CHALLENGE GOODWE+FIAP 2026.pdf — 14 páginas
Goodwe HCA Series Quick Installation Manual.pdf — 31 páginas
Goodwe HCA Series Quick Installation Manual 2.pdf — 31 páginas
GW_SEMS Portal APP_User Manual-EN.pdf — 33 páginas
GW_HCA-G2_Datasheet-EN.pdf — 2 páginas
GW_ESS Troubleshooting Guide-EMEA_EN.pdf — 37 páginas
Apresentação Challenge FIAP4+GoodWe 2026.pdf — 23 páginas
FIAP_EV Challenge_2026_Mentoria1.pdf — 11 páginas
Goodwe Hca Series User Manual.pdf — 68 páginas
GW_HCA Series_User Manual-EN.pdf — 46 páginas

432 chunk

In [18]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [19]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

print('Gerando embeddings...')

vectorstore = FAISS.from_documents(chunks, embeddings)

print(f'FAISS gerado — {vectorstore.index.ntotal} vetores')

Gerando embeddings...
FAISS gerado — 432 vetores


In [20]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.prompts import PromptTemplate

In [21]:
SYSTEM_PROMPT = """Você é um assistente técnico especializado em equipamentos de recarga
veicular da GoodWe. Você auxilia técnicos de campo na instalação, configuração,
diagnóstico de falhas e manutenção dos eletropostos HCA Series.

Regras:
- Responda com base no contexto fornecido abaixo.
- Mesmo que a informação seja parcial, responda com o que tiver disponível.
- Seja objetivo e técnico.
- Quando relevante, cite o documento de onde veio a informação.
- Responda em português brasileiro.
- Só diga que não encontrou a informação se o contexto realmente não contiver
  nada relacionado à pergunta.

Contexto:
{context}

Histórico:
{chat_history}

Pergunta: {question}
Resposta:"""

prompt = PromptTemplate(
    input_variables=['context', 'chat_history', 'question'],
    template=SYSTEM_PROMPT
)

llm = ChatOpenAI(
    model_name='gpt-4o-mini',
    temperature=0.1,
    max_tokens=1000
)

memory = ConversationBufferWindowMemory(
    k=5,
    memory_key='chat_history',
    return_messages=True,
    output_key='answer'
)

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)

chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={'prompt': prompt},
    return_source_documents=True,
    verbose=False
)

print('Chain configurada e pronta')

Chain configurada e pronta


In [22]:
TEST_QUESTIONS = [
    "The Led Charger is flashing green. What does that mean?",
    "What are the charging modes avaiable in the HCA series chargers?",
    "How to install the HCA Charger on the wall? What are the steps?",
    "How to configure the RFID card authentication on the HCA?",
    "How to connect the charger to the SolarGo app for the first time?"
]

print('=' * 70)
print('MODELO DE TESTE — SPRINT 1')
print('Chatbot GoodWe | Persona: Técnico de Campo')
print('=' * 70)

memory.clear()

results = []

for i, question in enumerate(TEST_QUESTIONS, 1):
    print(f'\n[PERGUNTA {i}]')
    print(f'>>> {question}')
    print()

    response = chain.invoke({'question': question})
    answer   = response['answer']
    sources  = response.get('source_documents', [])

    print(f'[RESPOSTA]')
    print(answer)

    if sources:
        print(f'\n[FONTES RECUPERADAS]')
        seen = set()
        for doc in sources:
            src = doc.metadata.get('source_file', 'desconhecido')
            page = doc.metadata.get('page', '?')
            key = f'{src} p.{page}'
            if key not in seen:
                print(f'   • {key}')
                seen.add(key)

    print('-' * 70)

    results.append({
        'pergunta': question,
        'resposta': answer,
        'fontes': [f"{d.metadata.get('source_file')} p.{d.metadata.get('page')}" for d in sources]
    })

    memory.clear()

MODELO DE TESTE — SPRINT 1
Chatbot GoodWe | Persona: Técnico de Campo

[PERGUNTA 1]
>>> The Led Charger is flashing green. What does that mean?

[RESPOSTA]
Quando o LED do carregador está piscando em verde, isso significa que o sistema do carregador está em processo de atualização. (Fonte: User Manual V1.3-2024-07-24, seção 3.6.3)

[FONTES RECUPERADAS]
   • GW_HCA Series_User Manual-EN.pdf p.20
   • GW_HCA Series_User Manual-EN.pdf p.38
   • Goodwe Hca Series User Manual.pdf p.23
   • Goodwe Hca Series User Manual.pdf p.42
----------------------------------------------------------------------

[PERGUNTA 2]
>>> What are the charging modes avaiable in the HCA series chargers?

[RESPOSTA]
Os modos de carregamento disponíveis nos carregadores da série HCA são: 

1. **Fast** (Rápido)
2. **PV priority** (Prioridade para energia solar)
3. **PV+battery** (Energia solar + bateria)

Esses modos permitem que o carregador utilize eletricidade da rede, de painéis solares (PV) ou de baterias para ca